# 2 - Training U-Net for Gap-Filling (Generic - Streaming)

**Self-supervised streaming training pipeline** for any Level-3 ocean variable.

This notebook trains a U-Net to fill gaps in satellite ocean observations using **xbatcher streaming** to keep memory bounded. Works for:
- Chlorophyll-a (PACE, Copernicus, CMEMS)
- Sea surface temperature
- Any other gridded ocean variable with cloud/missing data

## Key Features

- **Streaming workflow**: Zarr → xarray/Dask → xbatcher → TensorFlow
- **Memory-bounded**: Never loads full dataset into RAM
- **Spatial chunking**: 40×56 tiles (or configurable)
- **Generic target variable**: Not hardcoded to chlorophyll
- **Self-supervised**: No gap-free truth needed  
- **Synthetic clouds**: Temporally-correlated fake gaps for training/eval

## Workflow

```
Zarr (on-disk)
  ↓ xr.open_zarr with chunks
Lazy xarray/Dask Dataset
  ↓ build_standardized_lazy (no .load())
Lazy standardized channels
  ↓ xbatcher.BatchGenerator
Spatial tiles (time × 40×56)
  ↓ make_tf_gen → tf.data.Dataset
Training batches
  ↓
model.fit()
```

## Setup

In [ ]:
# Import the local checkout when running this notebook from the repository.
import sys
from pathlib import Path

for parent in (Path.cwd(), Path.cwd().parent):
    if (parent / "mindthegap").is_dir():
        sys.path.insert(0, str(parent))
        break

dataset = "globcolour"  # pace, globcolour, indian-ocean, or synthetic
region = "arabian sea"  # or [lat_min, lat_max, lon_min, lon_max]
time_slice = None

# Keep validation runs small. Set False for full training. demo_data applies
# the subsetting when smoke_test=True.
SMOKE_TEST = True


In [ ]:
# Icechunk is only required by the PACE and GlobColour loaders.
if dataset in {"pace", "globcolour"}:
    get_ipython().run_line_magic("pip", "install -q 'icechunk>=2'")

In [ ]:
if dataset in {"pace", "globcolour"}:
    import icechunk

    icechunk_major = int(icechunk.__version__.split(".", 1)[0])
    if icechunk_major < 2:
        raise RuntimeError("PACE and GlobColour require icechunk >= 2")
    print(f"Icechunk version: {icechunk.__version__}")

In [ ]:
# Earthaccess is only required by the authenticated PACE loader.
if dataset == "pace":
    get_ipython().run_line_magic("pip", "install -q 'earthaccess>=0.15'")

In [ ]:
if dataset == "pace":
    import earthaccess
    from packaging.version import Version

    if Version(earthaccess.__version__) < Version("0.15"):
        raise RuntimeError("PACE requires earthaccess >= 0.15")
    print(f"Earthaccess version: {earthaccess.__version__}")

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # Reduce TensorFlow verbosity

import numpy as np
import pandas as pd
import xarray as xr
import tensorflow as tf
import matplotlib.pyplot as plt
import mindthegap as mtg

# TensorFlow automatically uses CPU when no GPU is available.
gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

print(f"TensorFlow version: {tf.__version__}")
print(f"Compute device: {'GPU' if gpus else 'CPU'}")
print(f"GPUs available: {len(gpus)}")

## 1. Load Your Data

Load your xarray Dataset with **time, lat, lon** dimensions.

### Requirements:
- Target variable (e.g., `chlor_a`, `sst`, `analysed_sst`)
- Cloud/missing flag variable (1 = cloud/missing, 0 = valid data)
- Land flag variable (1 = land, 0 = ocean)
- Optional: Additional predictor variables (SST, winds, salinity, etc.)

### Data Source Examples:

In [ ]:
# Load the data. When SMOKE_TEST is set, demo_data returns a small subset.
ds, data_metadata = mtg.demo_data(
    dataset=dataset,
    region=region,
    time_slice=time_slice,
    smoke_test=SMOKE_TEST,
)
print(f"Selected dataset: {data_metadata['dataset']['name']}")
print(f"Dimensions: {dict(ds.sizes)}")


In [ ]:
# Smoke-test subsetting is applied inside mtg.demo_data(smoke_test=SMOKE_TEST).


## 2. Build Standardized Training data

Use `build_standardized_lazy()` to create standardized predictors **without loading data into memory**.

In [ ]:
# Crop to U-Net-compatible dimensions (multiples of 8).
ds = mtg.crop_to_multiple(ds, multiple=8)

print(f"\nAfter cropping to multiple of 8:")
print(f"Dimensions: {dict(ds.sizes)}")


In [ ]:
# Pipeline configuration
#
# `mtg.Options` is the single, canonical configuration for this run. Everything
# that can be inferred from the loaded dataset (variable names, bounds, tile
# size, batch size, training schedule, train/val split) is resolved by
# `set_data_config(data=ds)`. Downstream cells pass the section they own
# (options.gridder, options.fit, options.split, options.data) rather than
# threading individual arguments through functions.
options = mtg.Options.default()

# Preprocessing choices for the target (data-related config).
options.data.log_target = True     # Apply log to target? (True for chl, False for SST)
options.data.n_temporal_lags = 1   # Number of prev/next day channels

# Resolve all data-dependent configuration from the (cropped) dataset and its
# loader metadata in one call.
options.set_data_config(data=ds, metadata=data_metadata)

print(f"Dataset dimensions: {dict(ds.sizes)}")
print(f"Target variable: {options.data.target_variable}")
print(f"Date range: {pd.to_datetime(ds.time.values[0])} to {pd.to_datetime(ds.time.values[-1])}")
print()
print(options)


## Build lazy standardized dataset

* add n-prev and n-next targets as inputs
* add features (optional) as inputs; standardize
* add flags (land, real cloud, fake cloud)
* rename the target; standardize and log (optional)

In [ ]:
# Align output chunks with the resolved tile size for efficient xbatcher reads.
output_chunks = {
    "time": options.gridder.time_chunk,
    "lat": options.gridder.tile_size[0],
    "lon": options.gridder.tile_size[1],
}

# Variable names, features, log-transform, and lags all come from options.data;
# it is populated in place with the resolved channel order, standardization,
# and target mean/std.
ds_std, stats = mtg.build_standardized_lazy(
    ds,
    train_dates=options.split.train_slice(),
    std_vars=options.data.features,  # don't standardize target
    missing_flag_shift=10,
    output_chunks=output_chunks,
    add_geo=False,  # Set True to add spherical lat/lon features
    options=options.data,
)

num_channels = len(options.data.input_names)

print(f"\nChannels created ({num_channels} total):")
for i, ch in enumerate(options.data.input_names, 1):
    print(f"  {i}. {ch}")
print(f"\nTarget standardization: mean={options.data.target_mean:.4f}, "
      f"std={options.data.target_std:.4f}")
print(f"\nDataset is LAZY (not in memory): {ds_std.chunks}")
print(f"\nResolved data configuration:")
print(options.data)


## 3. Create xbatcher Streaming Pipeline

Use `mtg.make_xbatcher()` to create tile generators, then wrap with `mtg.make_tf_gen()` for TensorFlow.

In [ ]:
# Split data into train/val subsets using the resolved split.
ds_train = ds_std.sel(time=options.split.train_slice())
ds_val = ds_std.sel(time=options.split.val_slice())

print(f"\nTrain time range: {ds_train.time.values[0]} to {ds_train.time.values[-1]}")
print(f"Val time range: {ds_val.time.values[0]} to {ds_val.time.values[-1]}")

# Create xbatcher generators driven by the gridder configuration.
print("\nCreating xbatcher generators on in-memory data...")
train_batcher = mtg.make_xbatcher(ds_train, options=options.gridder)
val_batcher = mtg.make_xbatcher(ds_val, options=options.gridder)

print(f"Train tiles: {len(train_batcher)}")
print(f"Val tiles: {len(val_batcher)}")

# Calculate steps per epoch
tiles_per_batch = options.gridder.time_chunk
train_steps = (len(train_batcher) * tiles_per_batch) // options.fit.batch_size
val_steps = (len(val_batcher) * tiles_per_batch) // options.fit.batch_size
print(f"\nSteps per epoch:")
print(f"  Train: {train_steps}")
print(f"  Val: {val_steps}")


In [ ]:
# Wrap xbatcher generators with TensorFlow Dataset
print("Creating TensorFlow datasets...")

tile_lat, tile_lon = options.gridder.tile_size
num_channels = len(options.data.input_names)
output_signature = (
    tf.TensorSpec(shape=(tile_lat, tile_lon, num_channels), dtype=tf.float32),
    tf.TensorSpec(shape=(tile_lat, tile_lon, 1), dtype=tf.float32),
)

# Channel order and the target name both come from options.data.
train_dataset = tf.data.Dataset.from_generator(
    mtg.make_tf_gen(train_batcher, options.data.input_names, label=options.data.target),
    output_signature=output_signature
).shuffle(options.fit.shuffle_buffer).batch(options.fit.batch_size).repeat().prefetch(tf.data.AUTOTUNE)

val_dataset = tf.data.Dataset.from_generator(
    mtg.make_tf_gen(val_batcher, options.data.input_names, label=options.data.target),
    output_signature=output_signature
).batch(options.fit.batch_size).repeat().prefetch(tf.data.AUTOTUNE)

print("✓ TensorFlow datasets ready")
print(f"  Train: shuffle({options.fit.shuffle_buffer}) → batch({options.fit.batch_size}) → repeat → prefetch")
print(f"  Val: batch({options.fit.batch_size}) → repeat → prefetch")


## 4. Build U-Net Model

Fully-convolutional U-Net that can accept any spatial size (trains on 40×56, can predict on full domain).

In [ ]:
# Build U-Net (fully-convolutional). Compilation is handled by mtg.fit_model
# using options.fit so the training configuration lives in one place.
model = mtg.UNet((None, None, num_channels))

model.summary()

tile_lat, tile_lon = options.gridder.tile_size
print(f"\nModel input shape: (batch, {tile_lat}, {tile_lon}, {num_channels})")
print(f"Model output shape: (batch, {tile_lat}, {tile_lon}, 1)")


## 5. Train with Streaming Data

Train using xbatcher streaming—data is loaded one tile at a time, keeping memory bounded.

In [ ]:
# Train using the fit configuration. mtg.fit_model compiles the model with
# options.fit (optimizer, learning rate, loss) and installs an EarlyStopping
# callback using options.fit.patience.
print("Starting training...")
print(f"  Epochs: {options.fit.epochs}")
print(f"  Batch size: {options.fit.batch_size}")
print(f"  Steps per epoch: train={train_steps}, val={val_steps}")
print(f"  Early stopping patience: {options.fit.patience}")
print("\n" + "="*60)

history = mtg.fit_model(
    model,
    train_dataset,
    options.fit,
    validation_data=val_dataset,
    steps_per_epoch=train_steps,
    validation_steps=val_steps,
    verbose=1,
)

print(f"Best val_loss: {min(history.history['val_loss']):.6f}")
print(f"Final train_loss: {history.history['loss'][-1]:.6f}")


## 6. Visualize Training History

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4))

epochs_run = len(history.history['loss'])
ax.plot(history.history['loss'], label='Train Loss')
ax.plot(history.history['val_loss'], label='Val Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title(f'Training History ({epochs_run} epochs)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nBest epoch: {np.argmin(history.history['val_loss']) + 1}")
print(f"Best val_loss: {min(history.history['val_loss']):.6f}")

## 7. Test Prediction (Full Domain)

Load one test frame and predict on the full domain to verify the model works.

In [ ]:
# Load one test day for full-domain prediction
last_day = pd.to_datetime(ds.time.values[-1])
test_day = min(pd.to_datetime(options.split.val_end) + pd.Timedelta(days=15), last_day)
test_date = str(test_day.date())
print(f"Test prediction date: {test_date}")

# Select and load test frame
ds_test = ds_std.sel(time=test_date).load()

# Stack channels in the resolved order.
X_test = np.stack(
    [np.nan_to_num(ds_test[ch].values, nan=0.0) for ch in options.data.input_names],
    axis=-1,
).astype('float32')
X_test = X_test[np.newaxis, ...]  # Add batch dimension

print(f"Test input shape: {X_test.shape}")

# Predict (fully-convolutional model handles any size)
y_pred = model(X_test, training=False).numpy()[0, :, :, 0]

# Unstandardize using the resolved target statistics.
y_pred_orig = y_pred * options.data.target_std + options.data.target_mean

print(f"Prediction shape: {y_pred_orig.shape}")
print(f"Prediction range: [{np.nanmin(y_pred_orig):.4f}, {np.nanmax(y_pred_orig):.4f}]")
print("\n✓ Model successfully predicts on full domain")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True, sharey=True)

# Put observations in the same unstandardized units as the prediction.
observed = ds_test['masked_target'].values * options.data.target_std + options.data.target_mean
land = ds_test['land_flag'].values.astype(bool)
observed = np.ma.masked_where(land | ~np.isfinite(observed), observed)
prediction = np.ma.masked_where(land | ~np.isfinite(y_pred_orig), y_pred_orig)

# Use geographic coordinates and one robust scale for both panels.
vmin, vmax = np.nanpercentile(observed.filled(np.nan), [2, 98])
plot_kwargs = dict(cmap='viridis', shading='auto', vmin=vmin, vmax=vmax)
longitude = ds_test['lon'].values
latitude = ds_test['lat'].values

axes[0].set_facecolor('0.75')
axes[0].pcolormesh(longitude, latitude, observed, **plot_kwargs)
axes[0].set_title(f'Observed (with synthetic clouds)\n{test_date}')

axes[1].set_facecolor('0.75')
im = axes[1].pcolormesh(longitude, latitude, prediction, **plot_kwargs)
axes[1].set_title(f'U-Net Prediction\n{test_date}')

for ax in axes:
    ax.set_xlabel('Longitude (degrees east)')
    ax.set_ylabel('Latitude (degrees north)')
    ax.set_aspect('equal')

color_label = 'Log chlorophyll-a' if options.data.log_target else 'Chlorophyll-a'
plt.colorbar(im, ax=axes, label=color_label, fraction=0.02)
fig.subplots_adjust(wspace=0.08, right=0.88)
plt.show()


## 8. Save Model

Save the trained model for later use.

In [ ]:
# Everything the bundle needs is already resolved on options.data, so the
# metadata and the fitting pipeline cannot diverge. Save the complete resolved
# Options with the bundle.
region = {"lat": list(options.data.lat_bounds), "lon": list(options.data.lon_bounds)}
bundle_path = Path("../models") / f"{dataset}-unet-bundle"
metadata_path = mtg.create_model_bundle_metadata(
    bundle_path,
    model_name=f"{options.data.source} U-Net gap filler",
    dataset_name=options.data.source,
    product_id=options.data.product_id,
    region=region,
    training_period=options.data.training_period,
    input_names=options.data.input_names,
    target_name=options.data.target_name,
    target_units=options.data.target_units,
    expected_input_shape=list(model.input_shape),
    transforms=options.data.transforms,
    standardization=options.data.standardization,
    missing_value_handling=options.data.missing_value_handling,
    limitations=(
        "Validated only for the documented product, region, training period, "
        "channel order, and preprocessing configuration."
    ),
    options=options,
    overwrite=True,
)
print(f"Metadata saved to: {metadata_path}")


### Stop and review the metadata

Open the `model_metadata.yaml` file printed above. Check the dataset, region, training period, input channel order, transforms, and standardization values. **Do not run the next cell until the metadata is correct.**

In [ ]:
mtg.save_model_bundle(
    model,
    bundle_path,
    overwrite=True,
)
print(f"✓ Model bundle saved to: {bundle_path}")

In [ ]:
# Verify that the released artifact loads without rebuilding the U-Net.
loaded_model, loaded_metadata = mtg.load_model_bundle(bundle_path)
round_trip_input = X_test[:1]
expected = model(round_trip_input, training=False).numpy()
actual = loaded_model(round_trip_input, training=False).numpy()
np.testing.assert_allclose(actual, expected, rtol=1e-6, atol=1e-6)
assert [item["name"] for item in loaded_metadata["inputs"]] == options.data.input_names
print("✓ Bundle reload produced equivalent predictions")
